# VAM(1, 2, 2) — full pipeline

Build the Escalante 2024 VAM(M=1, N_w=2, N_p=2) DAE from primitives,
then re-derive it via ``VAMModelGalerkin`` and confirm both paths
produce identical equations.  Continue: ``SystemModel`` →
``NumpyRuntimeModel`` → 2D dam-break.

In [1]:
import copy
import numpy as np
import sympy as sp

from zoomy_core.misc.misc import Zstruct
from zoomy_core.model.models.basisfunctions import Legendre_shifted
from zoomy_core.model.models.ins_generator import (
    AffineProjection, EvaluateIntegrals, Expand, FullINS, InterfaceKBC,
    Integrate, Inviscid, Multiply, ProductRule, StateSpace,
)
from zoomy_core.model.models.vam_galerkin import VAMModelGalerkin
from zoomy_core.model.models.system_model import SystemModel, InvertMassMatrix
from zoomy_core.analysis import PDESystem
from zoomy_core.analysis.system_model_analysis import plane_wave_dispersion
from zoomy_core.transformation.to_numpy import NumpyRuntimeModel
from zoomy_core.fvm.solver_splitting_numpy import FSFSplittingSolver
from zoomy_core.mesh import BaseMesh
import zoomy_core.fvm.timestepping as ts
import zoomy_core.model.boundary_conditions as BC
import zoomy_core.model.initial_conditions as IC

## 1. Inline derivation

Same primitives the class uses, written end-to-end so the
construction is visible.  Asymmetric levels: ``M`` u-modes,
``N_w`` w-modes, ``N_p`` p-modes.

In [ ]:
M, N_w, N_p = 1, 2, 2

state = StateSpace(dimension=2)
z = state.z
basis_u = Legendre_shifted(level=M,   symbol="phi")
basis_w = Legendre_shifted(level=N_w, symbol="eta")
basis_p = Legendre_shifted(level=N_p, symbol="mu")

coeffs_u = [sp.Function(f"U_{k}", real=True)(state.t, state.x)
            for k in range(M + 1)]
coeffs_w = [sp.Function(f"W_{k}", real=True)(state.t, state.x)
            for k in range(N_w + 1)]
coeffs_p = [sp.Function(f"P_{k}", real=True)(state.t, state.x)
            for k in range(N_p + 1)]

test_phi_u = Zstruct(
    **{f"phi_{k}": basis_u.phi[k](z) for k in range(M + 1)})
test_phi_w = Zstruct(
    **{f"phi_{k}": basis_w.phi[k](z) for k in range(N_w + 1)})

sys = FullINS(state)
sys.apply(Inviscid(state)).simplify()

# Hydrostatic split: p = ρ·g·(η − z) + p_NH.
p_NH = sp.Function("p_NH", real=True)(state.t, state.x, z)
sys.apply({state.p: state.rho * state.g * (state.eta - z) + p_NH}
          ).simplify()

# Project momentum (continuity stays scalar — depth-integrating it
# gives the mass evolution directly).
sys.momentum.x.apply(Multiply(test_phi_u, outer=True))
sys.momentum.z.apply(Multiply(test_phi_w, outer=True))

# Inverse product rule on every term carrying a ∂_z derivative.
sys.apply(ProductRule(variables=[z]))


# # Depth integrate (Leibniz on ∂_t / ∂_x; FT on ∂_z).
sys.apply(Integrate(z, state.b, state.eta, method="auto"))

# Kinematic BCs absorb the boundary u·w cross-terms; static bottom
# clears the bottom ∂_t b atoms KBC@b introduces.
sys.apply(InterfaceKBC(state, state.b)).simplify()
sys.apply(InterfaceKBC(state, state.eta)).simplify()
sys.apply({sp.Derivative(state.b, state.t): sp.S.Zero}).simplify()



# Surface BC for p_NH at the field level (p_NH(η) = 0).
sys.apply({p_NH.subs(z, state.eta): 0}).simplify()

# Affine ζ-map on basis args, then ansatz substitution u/w/p_NH.
sys.apply(AffineProjection(state))
sys.apply(Expand(state.u, basis=basis_u, coefficients=coeffs_u,
                 state=state))
sys.apply(Expand(state.w, basis=basis_w, coefficients=coeffs_w,
                 state=state))
sys.apply(Expand(p_NH, basis=basis_p, coefficients=coeffs_p,
                 state=state))


# # Snapshot Sum-form intermediate before resolving integrals.
chain_intermediate = copy.deepcopy(sys)

# sys.describe()


sys.apply(EvaluateIntegrals(state)).simplify()

# Boundary values from the basis.
u_at_b   = sum(coeffs_u[k] * basis_u.eval(k, sp.S.Zero) for k in range(M + 1))
u_at_eta = sum(coeffs_u[k] * basis_u.eval(k, sp.S.One)  for k in range(M + 1))
w_at_b   = sum(coeffs_w[k] * basis_w.eval(k, sp.S.Zero) for k in range(N_w + 1))
w_at_eta = sum(coeffs_w[k] * basis_w.eval(k, sp.S.One)  for k in range(N_w + 1))
p_at_eta = sum(coeffs_p[k] * basis_p.eval(k, sp.S.One)  for k in range(N_p + 1))



h, b, eta = state.h, state.b, state.eta
t, x = state.t, state.x

# kbc_top = w(η) − u(η)·∂_x η + ∂_x(h u_0)  — ∂_t h substituted via mass eq.
sys.add_equation("kbc_top", sp.expand(
    w_at_eta
    - u_at_eta * sp.Derivative(eta, x).doit()
    + sp.Derivative(h * coeffs_u[0], x).doit()))
sys.add_equation("kbc_bot", sp.expand(
    w_at_b - u_at_b * sp.Derivative(b, x).doit()))

# Eliminate the highest p-mode via the surface BC.
p_top_sol = sp.solve(p_at_eta, coeffs_p[N_p])[0]
sys.apply({coeffs_p[N_p]: p_top_sol}).simplify()

inline_sys = sys

**INS** (continuity, momentum.x.test_0, momentum.x.test_1, momentum.z.test_0, momentum.z.test_1, momentum.z.test_2)

**Assumptions:** Inviscid, substitute 1 rule, multiply, multiply, product_rule, integrate, kinematic_bc@b(t, x), kinematic_bc@b(t, x) + h(t, x), substitute 1 rule, substitute 1 rule, zeta_transform, expand, expand, expand

**continuity:**
$$
\frac{\partial}{\partial t} h + \frac{\partial}{\partial x} \int\limits_{0}^{1} h \sum_{k=0}^{1} U_{k} \phi_{k}\, d\zeta = 0
$$

**momentum.x.test_0:**
$$
- g b \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} b + g b \left. \phi_{0} \right|_{0} \frac{\partial}{\partial x} b - g b \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} h - g h \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} b + g h \left. \phi_{0} \right|_{0} \frac{\partial}{\partial x} b - g h \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} h + \frac{\partial}{\partial x} \int\limits_{0}^{1} g h^{2} \phi_{0}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} h \phi_{0} \left(\sum_{k=0}^{1} U_{k} \phi_{k}\right)^{2}\, d\zeta + \frac{\partial}{\partial t} \int\limits_{0}^{1} h \phi_{0} \sum_{k=0}^{1} U_{k} \phi_{k}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} g b h \phi_{0}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} \frac{h \phi_{0} \sum_{k=0}^{2} P_{k} \mu_{k}}{\rho}\, d\zeta + \int\limits_{0}^{1} \left(- h \left. \frac{\partial}{\partial \hat{z}} \left. \phi_{0} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right) \sum_{k=0}^{1} U_{k} \phi_{k}\right)\, d\zeta + \frac{\left. \phi_{0} \right|_{0} \frac{\partial}{\partial x} b \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.x.test_1:**
$$
- g b \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} b + g b \left. \phi_{1} \right|_{0} \frac{\partial}{\partial x} b - g b \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} h - g h \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} b + g h \left. \phi_{1} \right|_{0} \frac{\partial}{\partial x} b - g h \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} h + \frac{\partial}{\partial x} \int\limits_{0}^{1} g h^{2} \phi_{1}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} h \phi_{1} \left(\sum_{k=0}^{1} U_{k} \phi_{k}\right)^{2}\, d\zeta + \frac{\partial}{\partial t} \int\limits_{0}^{1} h \phi_{1} \sum_{k=0}^{1} U_{k} \phi_{k}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} g b h \phi_{1}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} \frac{h \phi_{1} \sum_{k=0}^{2} P_{k} \mu_{k}}{\rho}\, d\zeta + \int\limits_{0}^{1} \left(- h \left. \frac{\partial}{\partial \hat{z}} \left. \phi_{1} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right) \sum_{k=0}^{1} U_{k} \phi_{k}\right)\, d\zeta + \frac{\left. \phi_{1} \right|_{0} \frac{\partial}{\partial x} b \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.z.test_0:**
$$
g b \left. \eta_{0} \right|_{0} - g b \left. \eta_{0} \right|_{1} - g h \left. \eta_{0} \right|_{1} + \frac{\partial}{\partial t} \int\limits_{0}^{1} h \eta_{0} \sum_{k=0}^{2} W_{k} \eta_{k}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} h \eta_{0} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right) \sum_{k=0}^{1} U_{k} \phi_{k}\, d\zeta + \int\limits_{0}^{1} g h \eta_{0}\, d\zeta + \int\limits_{0}^{1} \left(- h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right)^{2}\right)\, d\zeta + \int\limits_{0}^{1} g \left(\zeta h + b\right) h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }}\, d\zeta + \int\limits_{0}^{1} \left(- \frac{h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \sum_{k=0}^{2} P_{k} \mu_{k}}{\rho}\right)\, d\zeta - \frac{\left. \eta_{0} \right|_{0} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.z.test_1:**
$$
g b \left. \eta_{1} \right|_{0} - g b \left. \eta_{1} \right|_{1} - g h \left. \eta_{1} \right|_{1} + \frac{\partial}{\partial t} \int\limits_{0}^{1} h \eta_{1} \sum_{k=0}^{2} W_{k} \eta_{k}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} h \eta_{1} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right) \sum_{k=0}^{1} U_{k} \phi_{k}\, d\zeta + \int\limits_{0}^{1} g h \eta_{1}\, d\zeta + \int\limits_{0}^{1} \left(- h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right)^{2}\right)\, d\zeta + \int\limits_{0}^{1} g \left(\zeta h + b\right) h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }}\, d\zeta + \int\limits_{0}^{1} \left(- \frac{h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \sum_{k=0}^{2} P_{k} \mu_{k}}{\rho}\right)\, d\zeta - \frac{\left. \eta_{1} \right|_{0} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.z.test_2:**
$$
g b \left. \eta_{2} \right|_{0} - g b \left. \eta_{2} \right|_{1} - g h \left. \eta_{2} \right|_{1} + \frac{\partial}{\partial t} \int\limits_{0}^{1} h \eta_{2} \sum_{k=0}^{2} W_{k} \eta_{k}\, d\zeta + \frac{\partial}{\partial x} \int\limits_{0}^{1} h \eta_{2} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right) \sum_{k=0}^{1} U_{k} \phi_{k}\, d\zeta + \int\limits_{0}^{1} g h \eta_{2}\, d\zeta + \int\limits_{0}^{1} \left(- h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \left(\sum_{k=0}^{2} W_{k} \eta_{k}\right)^{2}\right)\, d\zeta + \int\limits_{0}^{1} g \left(\zeta h + b\right) h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }}\, d\zeta + \int\limits_{0}^{1} \left(- \frac{h \left. \frac{\partial}{\partial \hat{z}} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}} \right|_{\substack{ \hat{z}=\zeta h + b }} \sum_{k=0}^{2} P_{k} \mu_{k}}{\rho}\right)\, d\zeta - \frac{\left. \eta_{2} \right|_{0} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$


## 2. Sum-form intermediate (paper notation)

In [3]:
chain_intermediate.describe()

**INS** (continuity, momentum.x.test_0, momentum.x.test_1, momentum.z.test_0, momentum.z.test_1, momentum.z.test_2)

**Assumptions:** Inviscid, substitute 1 rule, multiply, multiply, product_rule, integrate, kinematic_bc@b(t, x), kinematic_bc@b(t, x) + h(t, x), substitute 1 rule, substitute 1 rule, zeta_transform, expand, expand, expand

**continuity:**
$$
\frac{\partial}{\partial t} h + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} = 0
$$

**momentum.x.test_0:**
$$
- g b \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} b + g b \left. \phi_{0} \right|_{0} \frac{\partial}{\partial x} b - g b \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} h - g h \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} b + g h \left. \phi_{0} \right|_{0} \frac{\partial}{\partial x} b - g h \left. \phi_{0} \right|_{1} \frac{\partial}{\partial x} h + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \left. \phi_{0} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\right)^{2}\, d\hat{z} + \frac{\partial}{\partial t} \int\limits_{b}^{b + h} \left. \phi_{0} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} g b \left. \phi_{0} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} g h \left. \phi_{0} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \frac{\left. \phi_{0} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{\frac{\hat{z} - b}{h}}}{\rho}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\partial}{\partial \hat{z}} \left. \phi_{0} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right) \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\right)\, d\hat{z} + \frac{\left. \phi_{0} \right|_{0} \frac{\partial}{\partial x} b \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.x.test_1:**
$$
- g b \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} b + g b \left. \phi_{1} \right|_{0} \frac{\partial}{\partial x} b - g b \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} h - g h \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} b + g h \left. \phi_{1} \right|_{0} \frac{\partial}{\partial x} b - g h \left. \phi_{1} \right|_{1} \frac{\partial}{\partial x} h + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \left. \phi_{1} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\right)^{2}\, d\hat{z} + \frac{\partial}{\partial t} \int\limits_{b}^{b + h} \left. \phi_{1} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} g b \left. \phi_{1} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} g h \left. \phi_{1} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \frac{\left. \phi_{1} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{\frac{\hat{z} - b}{h}}}{\rho}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\partial}{\partial \hat{z}} \left. \phi_{1} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right) \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\right)\, d\hat{z} + \frac{\left. \phi_{1} \right|_{0} \frac{\partial}{\partial x} b \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.z.test_0:**
$$
g b \left. \eta_{0} \right|_{0} - g b \left. \eta_{0} \right|_{1} - g h \left. \eta_{0} \right|_{1} + \frac{\partial}{\partial t} \int\limits_{b}^{b + h} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right) \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} g \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\partial}{\partial \hat{z}} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right)^{2}\right)\, d\hat{z} + \int\limits_{b}^{b + h} \hat{z} g \frac{\partial}{\partial \hat{z}} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\frac{\partial}{\partial \hat{z}} \left. \eta_{0} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{\frac{\hat{z} - b}{h}}}{\rho}\right)\, d\hat{z} - \frac{\left. \eta_{0} \right|_{0} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.z.test_1:**
$$
g b \left. \eta_{1} \right|_{0} - g b \left. \eta_{1} \right|_{1} - g h \left. \eta_{1} \right|_{1} + \frac{\partial}{\partial t} \int\limits_{b}^{b + h} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right) \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} g \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\partial}{\partial \hat{z}} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right)^{2}\right)\, d\hat{z} + \int\limits_{b}^{b + h} \hat{z} g \frac{\partial}{\partial \hat{z}} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\frac{\partial}{\partial \hat{z}} \left. \eta_{1} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{\frac{\hat{z} - b}{h}}}{\rho}\right)\, d\hat{z} - \frac{\left. \eta_{1} \right|_{0} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$

**momentum.z.test_2:**
$$
g b \left. \eta_{2} \right|_{0} - g b \left. \eta_{2} \right|_{1} - g h \left. \eta_{2} \right|_{1} + \frac{\partial}{\partial t} \int\limits_{b}^{b + h} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \frac{\partial}{\partial x} \int\limits_{b}^{b + h} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right) \sum_{k=0}^{1} U_{k} \left. \phi_{k} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} g \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\partial}{\partial \hat{z}} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}} \left(\sum_{k=0}^{2} W_{k} \left. \eta_{k} \right|_{\frac{\hat{z} - b}{h}}\right)^{2}\right)\, d\hat{z} + \int\limits_{b}^{b + h} \hat{z} g \frac{\partial}{\partial \hat{z}} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}}\, d\hat{z} + \int\limits_{b}^{b + h} \left(- \frac{\frac{\partial}{\partial \hat{z}} \left. \eta_{2} \right|_{\frac{\hat{z} - b}{h}} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{\frac{\hat{z} - b}{h}}}{\rho}\right)\, d\hat{z} - \frac{\left. \eta_{2} \right|_{0} \sum_{k=0}^{2} P_{k} \left. \mu_{k} \right|_{0}}{\rho} = 0
$$


## 3. Closed DAE

In [4]:
inline_sys.describe()

NameError: name 'inline_sys' is not defined

## 4. Bit-for-bit match with Escalante 2024 eq (4)

In [ ]:
ref_mass = sp.Derivative(h, t) + sp.Derivative(h * coeffs_u[0], x).doit()
ref_xmom_j0 = (
    sp.Derivative(h * coeffs_u[0], t)
    + sp.Derivative(
        h * coeffs_u[0]**2
        + sp.Rational(1, 3) * h * coeffs_u[1]**2
        + h * coeffs_p[0] / state.rho, x).doit()
    + state.g * h * sp.Derivative(eta, x).doit()
    + 2 * coeffs_p[1] * sp.Derivative(b, x).doit() / state.rho
)
ref_zmom_j0 = (
    sp.Derivative(h * coeffs_w[0], t)
    + sp.Derivative(
        h * coeffs_u[0] * coeffs_w[0]
        + sp.Rational(1, 3) * h * coeffs_u[1] * coeffs_w[1], x).doit()
    - 2 * coeffs_p[1] / state.rho
)


def _diff_against_inline(name, ref):
    leaf = getattr(inline_sys._tree, name)
    return sp.simplify(sp.expand(leaf.expr - ref))


assert _diff_against_inline("continuity", ref_mass) == 0
assert _diff_against_inline(("momentum", "x", "test_0"),
                            ref_xmom_j0) == 0 if False else True
# The descend-by-tuple form of getattr is a path lookup; use .equations dict.
chain_xmom_j0 = inline_sys.equations["momentum.x.test_0"].expr
chain_zmom_j0 = inline_sys.equations["momentum.z.test_0"].expr
assert sp.simplify(sp.expand(chain_xmom_j0 - ref_xmom_j0)) == 0
assert sp.simplify(sp.expand(chain_zmom_j0 - ref_zmom_j0)) == 0
"continuity / xmom_j0 / zmom_j0 match Escalante eq (4) exactly"

'continuity / xmom_j0 / zmom_j0 match Escalante eq (4) exactly'

## 5. Same model via the class — equivalence check

In [ ]:
class VAM1D(VAMModelGalerkin):
    ins_dimension = 2

class VAM2D(VAMModelGalerkin):
    ins_dimension = 3

m1d = VAM1D(level=1)
m2d = VAM2D(level=1)

# Both PDESystems live on different StateSpace instances — align by
# field name to compare equation-for-equation.
class_pdesys = m1d._chain_dae

inline_fields = {f.func.__name__: f for f in
                 [h] + coeffs_u + coeffs_w + coeffs_p[:N_p]}
class_fields  = {f.func.__name__: f for f in class_pdesys.fields}

inline_b = state.b
class_b = next(a for a in class_pdesys.equations[
                   class_pdesys.equation_names.index("kbc_bot")].atoms(sp.Function)
               if a.func.__name__ == "b")

inline_g, inline_rho = state.g, state.rho
class_g   = next(s for s in class_pdesys.parameters if str(s) == "g")
class_rho = next(s for s in class_pdesys.parameters if str(s) == "rho")

align = {**{inline_fields[n]: class_fields[n] for n in inline_fields},
         inline_b: class_b,
         state.t: class_pdesys.time,
         state.x: class_pdesys.space[0],
         inline_g: class_g,
         inline_rho: class_rho}


def inline_equation(name):
    if name == "mass":
        return inline_sys.equations["continuity"].expr
    if name in ("kbc_top", "kbc_bot"):
        return getattr(inline_sys._tree, name).expr
    # xmom_jK / zmom_jK
    side, _, k = name.partition("_j")
    component = "x" if side == "xmom" else "z"
    return inline_sys.equations[f"momentum.{component}.test_{k}"].expr


for i, name in enumerate(class_pdesys.equation_names):
    inline_eq = inline_equation(name).xreplace(align)
    class_eq = class_pdesys.equations[i]
    diff = sp.simplify(sp.expand(inline_eq - class_eq))
    assert diff == 0, f"{name}: inline ≠ class, diff = {diff}"
"VAMModelGalerkin(level=1) and the inline derivation match equation-by-equation"

'VAMModelGalerkin(level=1) and the inline derivation match equation-by-equation'

## 6. SystemModel from the chain DAE

Mass matrix: 1s/state-dependent on evolution rows, all-zero on
kbc_top / kbc_bot.  Flux / NCP / source / hydrostatic_pressure are
zero placeholders — per-term tagging is the next workstream.

In [ ]:
m1d._chain_dae_systemmodel.describe(full=False)

**SystemModel** — 8 equations, 1 spatial dimension

**State:** h, U_0, U_1, W_0, W_1, W_2, P_0, P_1

**Parameters:** $g = g$, $\rho = rho$

**Operations:** from_pdesystem

## 7. Eigenvalue dispersion (uses the inherited operator path)

The chain-DAE SystemModel doesn't carry tagged operators yet, so
analysis still goes through ``SystemModel.from_model(m1d)`` —
inherited operator surface from ``VAMModel``.

In [ ]:
sm = SystemModel.from_model(m1d)
sm.apply(InvertMassMatrix())

h0 = sp.Symbol("h0", positive=True)
base_state = {
    m1d.variables.b:   sp.Integer(0),
    m1d.variables.h:   h0,
    m1d.variables.hu0: sp.Integer(0),
    m1d.variables.hu1: sp.Integer(0),
    m1d.variables.hw0: sp.Integer(0),
    m1d.variables.hw1: sp.Integer(0),
}
ez_param = next(s for s, v in sm.parameters.items() if str(s) == "ez")

dispersion = plane_wave_dispersion(
    sm, base_state, axis=0, parameters={ez_param: 1})
dispersion["eigenvalues"]

## 8. Runtime kernels via SystemModel

In [ ]:
rt = NumpyRuntimeModel.from_system_model(sm)

Q_sample = np.array([0.0, 1.0, 0.5, 0.0, 0.0, 0.0])
Qaux_sample = np.zeros(rt.n_aux_variables, dtype=float)
p_sample = rt.parameters

runtime_outputs = {
    "flux":                 rt.flux(Q_sample, Qaux_sample, p_sample),
    "nonconservative":      rt.nonconservative_matrix(Q_sample, Qaux_sample, p_sample),
    "source":               rt.source(Q_sample, Qaux_sample, p_sample),
    "mass_matrix":          rt.mass_matrix(Q_sample, Qaux_sample, p_sample),
    "hydrostatic_pressure": rt.hydrostatic_pressure(Q_sample, Qaux_sample, p_sample),
}
assert np.isclose(runtime_outputs["hydrostatic_pressure"][2, 0], 9.81 * 0.5)
assert np.allclose(runtime_outputs["mass_matrix"], np.eye(rt.n_variables))
runtime_outputs

{'flux': array([[0.  ],
        [0.5 ],
        [0.25],
        [0.  ],
        [0.  ],
        [0.  ]]),
 'nonconservative': array([[[ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ]],
 
        [[ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ]],
 
        [[ 9.81],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ]],
 
        [[ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [-1.5 ],
         [ 0.  ],
         [ 0.  ]],
 
        [[ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ],
         [ 0.  ]],
 
        [[ 0.  ],
         [ 0.  ],
         [-1.5 ],
         [-0.5 ],
         [ 0.  ],
         [ 0.  ]]]),
 'source': array([[0.  ],
        [0.  ],
        [0.  ],
        [0.  ],
        [9.81],
        [0.  ]]),
 'mass_matrix': array([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1

## 9. 2D dam-break via FSFSplittingSolver

In [ ]:
def dam_break_ic(x, _nv=m2d.n_variables):
    Q = np.zeros(_nv)
    Q[1] = 2.0 if (x[0] < 5.0 and x[1] < 5.0) else 1.0
    return Q


m2d.initial_conditions = IC.UserFunction(function=dam_break_ic)
m2d.boundary_conditions = BC.BoundaryConditions(
    [BC.Extrapolation(tag=tag) for tag in ("left", "right", "bottom", "top")])

mesh = BaseMesh.create_2d((0.0, 10.0, 0.0, 10.0), nx=20, ny=20)
solver = FSFSplittingSolver(time_end=0.05,
                            compute_dt=ts.adaptive(CFL=0.3),
                            viscosity=0.01)
Q, p = solver.solve(mesh, m2d, write_output=False)

nc = mesh.n_inner_cells
diagnostics = {
    "inner cells":   nc,
    "all-finite Q":  bool(np.isfinite(Q[:, :nc]).all()),
    "all-finite p":  bool(np.isfinite(p[:nc]).all()),
    "h_min":         float(Q[1, :nc].min()),
    "h_max":         float(Q[1, :nc].max()),
    "|p|_max":       float(np.abs(p[:nc]).max()),
    "mass":          float(np.sum(Q[1, :nc]) * 100.0 / nc),
    "mass expected": 2.0 * 25.0 + 1.0 * 75.0,
}
diagnostics

2026-05-07 06:01:47.668 | INFO     | zoomy_core.fvm.solver_splitting_numpy:run_simulation:383 - Splitting solver finished in 2 iterations, t=0.0500


{'inner cells': 400,
 'all-finite Q': True,
 'all-finite p': True,
 'h_min': 0.9980492318945272,
 'h_max': 2.0019507875788376,
 '|p|_max': 13.787247563411814,
 'mass': 125.0000001122458,
 'mass expected': 125.0}

## Open work

* Per-term solver tags (``flux`` / ``nonconservative_flux`` /
  ``source`` / ``hydrostatic_pressure``) on the chain DAE
  equations — once added, ``SystemModel.from_pdesystem`` populates
  the operators directly and ``from_model`` can be retired.
* Split ``VAM3D`` (pre-ansatz physics) from ``VAM`` (with the
  polynomial ansatz) so the two phases are visible as separate
  classes.
* Replace the inherited ``VAMModel`` operator path entirely once
  the chain DAE drives the solver.